# Baseline: GRU4Rec

Hidasi & Karatzoglou (2018), con la implementación oficial en PyTorch ([GRU4Rec_PyTorch_Official](https://github.com/hidasib/GRU4Rec_PyTorch_Official)) entrenada con el protocolo de LatentR3: Adam, grilla de lr × weight decay, BCE con negativos aleatorios (Apéndice B) y contexto de hasta 10 ítems (Apéndice A). El detalle de las adaptaciones está en `README.md`.

En local lee los datos desde `../repo_latentr3/data` y el código oficial desde `../GRU4Rec_PyTorch_Official`; en Colab, desde `data_amazon_2018/` y `GRU4Rec_PyTorch_Official/` en el directorio de trabajo. No escribe archivos.

## Importaciones

In [19]:
import os
import io
import sys
import ast
import time
import random
import itertools
import contextlib
import numpy as np
import pandas as pd
import torch

In [20]:
SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

def fijar_semillas():
    # Se llama antes de cada combinación, para que cada una parta del mismo estado aleatorio
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

cuda AMD Radeon RX 9070 XT


## Datos y código oficial

In [21]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


In [22]:
base = 'data_amazon_2018' if IN_COLAB else os.path.join('..', 'repo_latentr3', 'data')
codigo = 'GRU4Rec_PyTorch_Official' if IN_COLAB else os.path.join('..', 'GRU4Rec_PyTorch_Official')
names = {"CDs_and_Vinyl": "cd_data",
         "Musical_Instruments": "music_data",
         "Toys_and_Games": "toys_data",
         "Video_Games": "games_data"}

if not os.path.isfile(os.path.join(codigo, "gru4rec_pytorch.py")):
    raise FileNotFoundError(f"No se encontró gru4rec_pytorch.py en {os.path.abspath(codigo)}. "
                            "Descargar https://github.com/hidasib/GRU4Rec_PyTorch_Official en esa carpeta.")
sys.path.append(codigo)
from gru4rec_pytorch import GRU4Rec, GRU4RecModel, SessionDataIterator, SampleCache

In [ ]:
MAX_LEN = 10   # contexto máximo del paper (Apéndice A)

def buscar_archivo(split, categoria):
    carpeta = os.path.join(base, split)
    archivos = [f for f in os.listdir(carpeta) if f.startswith(categoria)]
    if not archivos:
        raise FileNotFoundError(f"No hay un archivo de {categoria} en {os.path.abspath(carpeta)}")
    return os.path.join(carpeta, archivos[0])

def cargar(split, categoria):
    df = pd.read_csv(buscar_archivo(split, categoria))
    df["history_item_id"] = df["history_item_id"].apply(ast.literal_eval)
    return df

def cargar_catalogo(categoria):
    with open(buscar_archivo("info", categoria), encoding="utf-8") as f:
        ids = sorted(int(linea.rstrip("\n").rsplit("\t", 1)[1]) for linea in f if linea.strip())
    return pd.Series(np.arange(len(ids), dtype="int32"), index=ids, name="ItemIdx")

def a_sesiones(df):
    filas, sid = [], 0
    for _, g in df.groupby("user_id", sort=False):
        seq = g["history_item_id"].iloc[0] + g["item_id"].tolist()
        for fin in reversed(range(len(seq), 1, -MAX_LEN)):
            bloque = seq[max(0, fin - MAX_LEN - 1):fin]
            filas += [(sid, item, t) for t, item in enumerate(bloque)]   # Time es la posición dentro del bloque
            sid += 1
    return pd.DataFrame(filas, columns=["SessionId", "ItemId", "Time"])

datos = {categoria: {split: cargar(split, categoria) for split in ["train", "valid", "test"]}
         for categoria in names}
catalogos = {categoria: cargar_catalogo(categoria) for categoria in names}
print({names[c]: {**{s: len(df) for s, df in d.items()}, "ítems": len(catalogos[c])} for c, d in datos.items()})

{'cd_data': {'train': 49251, 'valid': 6156, 'test': 6157, 'ítems': 5841}, 'music_data': {'train': 66500, 'valid': 8312, 'test': 8313, 'ítems': 5030}, 'toys_data': {'train': 53898, 'valid': 6737, 'test': 6738, 'ítems': 6299}, 'games_data': {'train': 75175, 'valid': 9397, 'test': 9397, 'ítems': 5308}}


## Métricas

In [24]:
def hit(recs, objetivo, k):
    return 1 if objetivo in list(recs[:k]) else 0

def ndcg(recs, objetivo, k):
    recs = list(recs[:k])
    if objetivo not in recs:
        return 0
    pos = recs.index(objetivo)
    return 1 / np.log2(pos + 2)

def evaluar(recomendaciones, test_df, ks=[5, 10]):
    resultados = {}
    for k in ks:
        resultados[f"HR@{k}"] = np.mean([hit(r, o, k) for r, o in zip(recomendaciones, test_df["item_id"])])
        resultados[f"NDCG@{k}"] = np.mean([ndcg(r, o, k) for r, o in zip(recomendaciones, test_df["item_id"])])
    return resultados

## Modelo GRU4Rec (implementación oficial)

**No lo especifica el paper:** capa de 100, batch de 64, sin dropout y embedding compartido (valores por defecto oficiales); un negativo uniforme por fila; hasta 100 épocas con early stopping por NDCG@10 en `valid` (protocolo común de los baselines tradicionales); ranking sobre todo el catálogo de `info/`, igual que los métodos con LLM.

**Adaptaciones al código oficial**, que se usa sin modificar (`GRU4RecPaper` hereda de `GRU4Rec`):
- **Pérdida:** BCE en vez de cross-entropy o BPR-max, como pide el paper.
- **Catálogo:** `itemidmap` con todos los ítems de `info/` en vez de solo los de train, para que todos los métodos compitan sobre los mismos candidatos.
- **Negativos:** uno uniforme por fila sobre todo el catálogo (un `SampleCache` oficial con distribución uniforme y `n_sample = batch_size`), en vez de 2048 por popularidad compartidos. El muestreo por popularidad del iterador no admite ítems ausentes de train.
- **Optimizador:** Adam con weight decay y embeddings densos, en vez de Adagrad disperso. Adam no acepta gradientes dispersos.
- **Épocas:** early stopping en `valid` en vez de 10 épocas fijas, que están pensadas para Adagrad con lr altas.
- **Contexto:** `a_sesiones` corta la secuencia de cada usuario en bloques de hasta `MAX_LEN + 1` ítems desde el final, con el estado oculto en cero al inicio de cada bloque. Respeta el máximo de 10 sin duplicar transiciones.
- **Evaluación:** `recomendar` pasa el historial (≤10) de cada fila desde un estado cero y devuelve el top-k sobre todo el catálogo.

In [25]:
@torch.no_grad()
def recomendar(gru, df, k=10):
    item_a_idx = gru.data_iterator.itemidmap.to_dict()
    idx_a_item = gru.data_iterator.itemidmap.index.values
    hists = [[item_a_idx[i] for i in h if i in item_a_idx] for h in df["history_item_id"]]
    recs = [[] for _ in hists]
    for largo in set(map(len, hists)) - {0}:
        filas = [j for j, h in enumerate(hists) if len(h) == largo]
        X = torch.tensor([hists[j] for j in filas], device=device)
        H = [torch.zeros(len(filas), n, device=device) for n in gru.layers]
        for t in range(largo):
            scores = gru.model.forward(X[:, t], H, None, training=False)
        top = scores.topk(k, dim=1).indices.cpu().numpy()
        for j, r in zip(filas, top):
            recs[j] = idx_a_item[r].tolist()
    return recs

In [ ]:
class GRU4RecPaper(GRU4Rec):
    def __init__(self, weight_decay, **kwargs):
        super().__init__(**kwargs)
        self.weight_decay = weight_decay

    def set_loss_function(self, loss):
        self.loss_function = self.bce_loss

    def bce_loss(self, O, Y, M):
        n = O.shape[0]
        positivos = torch.diag(O[:, :n])
        negativos = torch.diag(O[:, n:])
        return -(torch.nn.functional.logsigmoid(positivos).sum()
                 + torch.nn.functional.logsigmoid(-negativos).sum())

    def fit(self, data, itemidmap, valid_df, max_epocas, paciencia):
        with contextlib.redirect_stdout(io.StringIO()):
            self.data_iterator = SessionDataIterator(data, self.batch_size, n_sample=0, device=self.device,
                                                     itemidmap=itemidmap)
            n_items = len(itemidmap)
            uniforme = torch.arange(1, n_items + 1, dtype=torch.float32, device=self.device) / n_items
            self.data_iterator.sample_cache = SampleCache(self.n_sample, 10000000, uniforme, device=self.device)
        self.model = GRU4RecModel(self.data_iterator.n_items, self.layers, self.dropout_p_embed, self.dropout_p_hidden,
                                  self.embedding, self.constrained_embedding).to(self.device)
        for modulo in self.model.modules():
            if isinstance(modulo, torch.nn.Embedding):
                modulo.sparse = False
        opt = torch.optim.Adam(self.model.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay)
        mejor, sin_mejora = {"ndcg": -1}, 0
        for epoca in range(max_epocas):
            H = [torch.zeros((self.batch_size, n), device=self.device) for n in self.layers]
            reset_hook = lambda n_valid, finished_mask, valid_mask: self._adjust_hidden(n_valid, finished_mask, valid_mask, H)
            for in_idx, out_idx in self.data_iterator(enable_neg_samples=True, reset_hook=reset_hook):
                for h in H:
                    h.detach_()
                self.model.zero_grad()
                R = self.model.forward(in_idx, H, out_idx, training=True)
                L = self.loss_function(R, out_idx, self.batch_size) / self.batch_size
                L.backward()
                opt.step()
            ndcg10 = evaluar(recomendar(self, valid_df), valid_df)["NDCG@10"]
            if ndcg10 > mejor["ndcg"]:
                estado = {k: v.detach().clone() for k, v in self.model.state_dict().items()}
                mejor, sin_mejora = {"ndcg": ndcg10, "epoca": epoca + 1, "estado": estado}, 0
            else:
                sin_mejora += 1
                if sin_mejora >= paciencia:
                    break
        self.model.load_state_dict(mejor["estado"])
        return mejor["epoca"], mejor["ndcg"]

## Grilla y evaluación

Por dataset se entrenan las 9 combinaciones de lr ∈ {1e-2, 1e-3, 1e-4} × weight decay ∈ {1e-4, 1e-5, 1e-6}. Se elige la de mejor NDCG@10 en `valid` y solo esa se evalúa en `test` con HR y NDCG @5 y @10, comparables con la Tabla 1 del paper. Se imprime una línea por combinación y al final la tabla de resultados.

In [ ]:
MAX_EPOCAS = 100
PACIENCIA = 10
LRS = [1e-2, 1e-3, 1e-4]
WEIGHT_DECAYS = [1e-4, 1e-5, 1e-6]
BATCH_SIZE = 64
FIJOS = dict(layers=[100], batch_size=BATCH_SIZE, n_sample=BATCH_SIZE, dropout_p_embed=0.0, dropout_p_hidden=0.0,
             logq=0.0, constrained_embedding=True, embedding=0)

filas = []
inicio = time.time()
for categoria, nombre in names.items():
    sesiones = a_sesiones(datos[categoria]["train"])
    valid_df, test_df = datos[categoria]["valid"], datos[categoria]["test"]
    mejor = None
    for lr, wd in itertools.product(LRS, WEIGHT_DECAYS):
        fijar_semillas()
        gru = GRU4RecPaper(weight_decay=wd, loss="bce", learning_rate=lr, device=device, **FIJOS)
        epoca, ndcg_valid = gru.fit(sesiones, catalogos[categoria], valid_df, MAX_EPOCAS, PACIENCIA)
        print(f"{nombre} lr={lr} wd={wd}: NDCG@10 valid {ndcg_valid:.4f} (época {epoca}) "
              f"[{(time.time() - inicio) / 60:.1f} min]", flush=True)
        if mejor is None or ndcg_valid > mejor["NDCG@10 valid"]:
            mejor = {"dataset": nombre, "lr": lr, "weight_decay": wd, "epoca": epoca,
                     "NDCG@10 valid": ndcg_valid, **evaluar(recomendar(gru, test_df), test_df)}
    filas.append(mejor)

resultados = pd.DataFrame(filas)
resultados.round({m: 4 for m in ["NDCG@10 valid", "HR@5", "NDCG@5", "HR@10", "NDCG@10"]})

cd_data lr=0.01 wd=0.0001: NDCG@10 valid 0.0168 (época 9) [0.3 min]
cd_data lr=0.01 wd=1e-05: NDCG@10 valid 0.0187 (época 28) [0.9 min]
cd_data lr=0.01 wd=1e-06: NDCG@10 valid 0.0137 (época 10) [1.2 min]
cd_data lr=0.001 wd=0.0001: NDCG@10 valid 0.0221 (época 77) [2.5 min]
cd_data lr=0.001 wd=1e-05: NDCG@10 valid 0.0416 (época 75) [3.8 min]
cd_data lr=0.001 wd=1e-06: NDCG@10 valid 0.0484 (época 100) [5.3 min]
cd_data lr=0.0001 wd=0.0001: NDCG@10 valid 0.0074 (época 3) [5.5 min]
cd_data lr=0.0001 wd=1e-05: NDCG@10 valid 0.0075 (época 1) [5.7 min]
cd_data lr=0.0001 wd=1e-06: NDCG@10 valid 0.0076 (época 1) [5.9 min]
music_data lr=0.01 wd=0.0001: NDCG@10 valid 0.0618 (época 3) [6.1 min]
music_data lr=0.01 wd=1e-05: NDCG@10 valid 0.0610 (época 30) [6.9 min]
music_data lr=0.01 wd=1e-06: NDCG@10 valid 0.0447 (época 8) [7.3 min]
music_data lr=0.001 wd=0.0001: NDCG@10 valid 0.0679 (época 50) [8.5 min]
music_data lr=0.001 wd=1e-05: NDCG@10 valid 0.0751 (época 32) [9.4 min]
music_data lr=0.001 wd

,dataset,lr,weight_decay,epoca,NDCG@10 valid,HR@5,NDCG@5,HR@10,NDCG@10
0,cd_data,0.001,0.000001,100,0.0484,0.0607,0.0444,0.0801,0.0506
1,music_data,0.001,0.000001,49,0.0885,0.0806,0.0702,0.0978,0.0757
2,toys_data,0.001,0.000010,100,0.0511,0.0435,0.0302,0.0657,0.0374
3,games_data,0.001,0.000010,41,0.0461,0.0345,0.0222,0.0568,0.0294
